In [0]:
# Extracting unique dates and calculating time attributes
# Using the purchase timestamp as the grain for the dimension
query = """
SELECT
    CAST(date_format(date, 'yyyyMMdd') AS INT) AS date_key,
    date,
    year(date) AS year,
    month(date) AS month,
    day(date) AS day,
    date_format(date, 'MMMM') AS month_name,
    date_format(date, 'EEEE') AS day_name,
    CASE
        WHEN month(date) BETWEEN 1 AND 3 THEN 'Q1'
        WHEN month(date) BETWEEN 4 AND 6 THEN 'Q2'
        WHEN month(date) BETWEEN 7 AND 9 THEN 'Q3'
        ELSE 'Q4'
    END AS quarter
FROM (
    SELECT DISTINCT CAST(order_purchase_timestamp AS DATE) AS date
    FROM workspace.silver.orders
)
WHERE date IS NOT NULL
ORDER BY date
"""
df = spark.sql(query)

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.dim_date")
)

In [0]:
%sql

SELECT 
 *
FROM workspace.gold.dim_date